In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from notebooks.utils.classification_experiment_utils import (
    DATASETS, EVALUATION_REGIMES, TRAINING_REGIMES, availability_row,
    build_feature_parent_cohorts, discover_model_artifacts, evaluate_feature_vector_model, export_result_frames,
    load_episode_partitions,
    release_frames,
)
from notebooks.utils.feature_aggregation_utils import load_prepared_partition, prepared_partition_path
from notebooks.utils.model_training_utils import load_model_artifact, load_model_metadata

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
MODELS_ROOT = DATA_ROOT / "experiment-results" / "classification" / "models"
RUN_ID = "feature-vector-classification-evaluation-2026-09-11"
SCORER_LLMS = ("gemma_3_1b_it", "gemma_3_270m_it", "littlelamb", "smollm2_360m")
SOURCE_SPLITS = {"Kaggle": ("HG", "MG"), "MAGE": ("HG", "MG"), "Outfox": ("HG", "MG"), "BBC": ("HG",)}
EVALUATION_WINDOWS = ("whole_text", 32, 64, 128)
MODEL_FAMILIES = ("logistic_regression",)

# Evaluation never infers a schema: artifact metadata supplies every transfer dimension.
artifact_paths = discover_model_artifacts(MODELS_ROOT, filters={})


In [ ]:
evaluation_metrics, evaluation_predictions, evaluation_coverage, run_summary = [], [], [], []
for artifact_path in tqdm(artifact_paths, desc="Evaluating classifier artifacts", unit="artifact"):
    try:
        metadata = load_model_metadata(artifact_path, expected={})
        scorer_llm = str(metadata["scorer_llm"])
        feature_set_name = str(metadata["feature_set"])
        feature_columns = tuple(metadata["feature_columns"])
        model_family = str(metadata["model_family"])
        if scorer_llm not in SCORER_LLMS or model_family not in MODEL_FAMILIES:
            continue
        fitted = load_model_artifact(artifact_path, expected={"model_family": model_family, "scorer_llm": scorer_llm, "feature_set": feature_set_name, "feature_columns": list(feature_columns)}, trust_artifact=True)
    except (FileNotFoundError, KeyError, ValueError, OSError) as error:
        run_summary.append({"run_id": RUN_ID, "stage": "evaluation", "status": "unavailable", "scorer_llm": "", "dataset": "", "split": "", "feature_set": "", "training_window": "", "evaluation_window": "", "model_family": "", "artifact_path": str(artifact_path), "error": str(error)})
        continue
    except Exception:
        raise

    training_regime = str(metadata["training_regime"])
    training_window = str(metadata["training_window"])
    for evaluation_regime, source_names in tqdm(EVALUATION_REGIMES.items(), desc="Evaluation regimes", unit="regime", leave=False):
        for evaluation_window in tqdm(EVALUATION_WINDOWS, desc=f"{evaluation_regime} windows", unit="window", leave=False):
            frames, parent_frame = {}, None
            try:
                for source_name in source_names:
                    dataset = DATASETS[source_name]
                    split_frames = []
                    for split in SOURCE_SPLITS[source_name]:
                        path = prepared_partition_path(DATA_ROOT, scorer_llm=scorer_llm, dataset=dataset, split=split, feature_set=feature_set_name, window_configuration=evaluation_window)
                        expected = {"schema_version": 1, "scorer_llm": scorer_llm, "dataset": dataset, "split": split, "feature_set": feature_set_name, "feature_columns": list(feature_columns), "window_configuration": "whole_text" if evaluation_window == "whole_text" else str(evaluation_window), "stride": None if evaluation_window == "whole_text" else 16}
                        split_frames.append(load_prepared_partition(path, expected=expected).assign(dataset=dataset, scorer_llm=scorer_llm, feature_set=feature_set_name))
                    frames[dataset] = pd.concat(split_frames, ignore_index=True)
                parent_frame = pd.concat([frame[["dataset", "split", "text_id", "label"]].assign(dataset=dataset) for dataset, frame in frames.items()], ignore_index=True)
                cohorts = build_feature_parent_cohorts(parent_frame, seed=int(metadata.get("cohort_seed", 42)), max_per_split=int(metadata.get("max_per_split", 500)))
                validation_cohort, test_cohort = cohorts["validation"], cohorts["test"]
                release_frames(parent_frame, *frames.values())
                frames = {}
                metadata_for_run = {**metadata, "training_window": training_window, "evaluation_window": "whole_text" if evaluation_window == "whole_text" else str(evaluation_window), "evaluation_regime": evaluation_regime}
                for partition in ("validation", "test"):
                    selected_cohort = validation_cohort if partition == "validation" else test_cohort
                    frames = load_episode_partitions(DATA_ROOT, scorer_llm=scorer_llm, datasets=tuple(DATASETS[name] for name in source_names), partition=partition, feature_set=feature_set_name, window_configuration=evaluation_window, feature_columns=feature_columns, cohort=selected_cohort)
                    metrics, predictions, coverage = evaluate_feature_vector_model(fitted, frames, partition=partition, metadata=metadata_for_run)
                    evaluation_metrics.extend(metrics.to_dict(orient="records"))
                    evaluation_predictions.append(predictions)
                    evaluation_coverage.extend(coverage.to_dict(orient="records"))
                    run_summary.append({"run_id": RUN_ID, "stage": "evaluation", "status": "available", "scorer_llm": scorer_llm, "dataset": ", ".join(source_names), "split": partition, "feature_set": feature_set_name, "training_window": training_window, "evaluation_window": metadata_for_run["evaluation_window"], "model_family": model_family, "artifact_path": str(artifact_path), "error": ""})
            except (FileNotFoundError, KeyError, ValueError, OSError) as error:
                run_summary.append({"run_id": RUN_ID, "stage": "evaluation", "status": "unavailable", "scorer_llm": scorer_llm, "dataset": evaluation_regime, "split": "", "feature_set": feature_set_name, "training_window": training_window, "evaluation_window": str(evaluation_window), "model_family": model_family, "artifact_path": str(artifact_path), "error": str(error)})
                continue
            except Exception:
                raise
            finally:
                release_frames(parent_frame, *frames.values())
    release_frames(fitted)


In [ ]:
evaluation_predictions_frame = pd.concat(evaluation_predictions, ignore_index=True) if evaluation_predictions else pd.DataFrame()
exports = export_result_frames(DATA_ROOT, run_id=RUN_ID, frames={
    "training_metrics": pd.DataFrame(),
    "training_coverage": pd.DataFrame(),
    "evaluation_metrics": pd.DataFrame(evaluation_metrics),
    "evaluation_predictions": evaluation_predictions_frame,
    "evaluation_coverage": pd.DataFrame(evaluation_coverage),
    "run_summary": pd.DataFrame(run_summary),
})
exports
